> **Deprecated.** Superseded by `kaggriculture-self-training/kaggriculture-self-training.ipynb` (Path B hierarchical DQN + bootstrap + self-play). This notebook used flat 10D obs / 9-action DQN only.


## Setup & Installations

Input: Kaggle notebook environment.
Output: Installed dependencies (kaggle-environments, stable-baselines3).
Expected Behavior: Environment is prepared for simulation and RL training.

In [ ]:
# Cell 1: Installations & Imports
!pip install -q "kaggle-environments>=1.32.2" "stable-baselines3[extra]" gymnasium kagglehub pandas

import os
import base64
import glob
import json
import random
from pathlib import Path

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium import spaces
from kaggle_environments import make
from stable_baselines3.common.buffers import ReplayBuffer

## Unified Observation & Action Helpers

Shared 10D state encoding and Discrete(9) action mapping used for replay parsing, online training, and `agent.py` generation.

In [ ]:
# Unified 10D observation + Discrete(9) action helpers
ACTION_NAMES = [
    "PASS", "NORTH", "SOUTH", "EAST", "WEST",
    "WATER", "HARVEST", "PLANT_MELON", "SELL_MELON",
]

def obs_to_dict(obs):
    if obs is None:
        return {}
    if isinstance(obs, dict):
        return obs
    try:
        return dict(obs)
    except (TypeError, ValueError):
        pass
    out = {}
    for key in ("player", "day", "hour", "step", "farms", "market", "private", "town"):
        if hasattr(obs, key):
            out[key] = getattr(obs, key)
    return out

def parse_obs(obs, player_id=0):
    obs = obs_to_dict(obs)
    farms = obs.get("farms", [])
    if not farms or len(farms) <= player_id:
        return np.zeros(10, dtype=np.float32)

    farm = farms[player_id]
    fx, fy = farm.get("farmer", [0, 0])
    money = farm.get("money", 0)

    opp_id = 1 - player_id
    opp_money = farms[opp_id].get("money", 0) if len(farms) > opp_id else 0

    market = obs.get("market", {}) or {}
    prices = market.get("prices", {}) or {}
    melon_price = prices.get("MELON", 0)

    market_inv = market.get("inventory", {}) or {}
    melon_market_inv = market_inv.get("MELON", 0)

    private = obs.get("private", {}) or {}
    shed = private.get("shed", {}) or {}
    melon_inv = shed.get("MELON", 0)

    seeds = private.get("seeds", {}) or {}
    melon_seeds = seeds.get("MELON", 0)

    day = obs.get("day", 0)
    hour = obs.get("hour", 0)

    return np.array([
        float(fx),
        float(fy),
        float(money) / 3000.0,
        float(opp_money) / 3000.0,
        float(melon_price) / 100.0,
        float(melon_market_inv) / 100.0,
        float(melon_inv) / 10.0,
        float(melon_seeds) / 10.0,
        float(day) / 30.0,
        float(hour) / 24.0,
    ], dtype=np.float32)

def parse_action_from_replay(action_dict):
    """Map native farmer/market/hands replay actions to Discrete(9)."""
    if not action_dict:
        return 0

    market_act = action_dict.get("market") or []
    for order in market_act:
        if isinstance(order, (list, tuple)) and len(order) >= 2:
            if order[0] == "SELL" and order[1] == "MELON":
                return 8

    farmer_act = action_dict.get("farmer") or []
    if isinstance(farmer_act, str):
        farmer_act = [farmer_act]
    if not farmer_act:
        return 0

    op = farmer_act[0]
    if op in ("NORTH", "MOVE_NORTH"):
        return 1
    if op in ("SOUTH", "MOVE_SOUTH"):
        return 2
    if op in ("EAST", "MOVE_EAST"):
        return 3
    if op in ("WEST", "MOVE_WEST"):
        return 4
    if op == "WATER":
        return 5
    if op == "HARVEST":
        return 6
    if op == "PLANT":
        crop = farmer_act[1] if len(farmer_act) > 1 else None
        return 7 if crop in (None, "MELON") else 0
    return 0

def format_action_for_env(action_idx):
    actions = [
        {"farmer": ["PASS"], "hands": [], "market": []},
        {"farmer": ["NORTH"], "hands": [], "market": []},
        {"farmer": ["SOUTH"], "hands": [], "market": []},
        {"farmer": ["EAST"], "hands": [], "market": []},
        {"farmer": ["WEST"], "hands": [], "market": []},
        {"farmer": ["WATER"], "hands": [], "market": []},
        {"farmer": ["HARVEST"], "hands": [], "market": []},
        {"farmer": ["PLANT", "MELON"], "hands": [], "market": [["BUY_SEED", "MELON", 1]]},
        {"farmer": ["PASS"], "hands": [], "market": [["SELL", "MELON", 1]]},
    ]
    return actions[int(action_idx)]

def to_replay_action(action_idx):
    """SB3 ReplayBuffer expects numpy actions, not Python ints."""
    return np.array(int(action_idx), dtype=np.int64)

print("Unified helpers ready:", ACTION_NAMES)

## Dataset Integration (Episodes Index)

Input: The daily top episodes Kaggle dataset URL/API.
Output: Downloaded replay JSON files (with local `replays/` fallback).

In [ ]:
# Cell 2: Download Top Episodes for Replay Buffer Bootstrapping
import kagglehub

latest_episodes = None
try:
    path = kagglehub.dataset_download("kaggle/kaggriculture-episodes-index")
    manifest = pd.read_csv(f"{path}/manifest.csv")
    latest_episodes = manifest.sort_values("date", ascending=False).iloc[0]["daily_dataset_slug"]
    print(f"Latest Episodes for training reference: {latest_episodes}")
except Exception as exc:
    print(f"Could not load episodes index ({exc}); will try local replays/ only.")

## Seeding the Replay Buffer

Download episode JSON files, parse simulation steps, and extract transitions into a Stable Baselines3 ReplayBuffer.

In [ ]:
# 1. Define spaces matching our architecture (10D Observation, 9-Discrete Actions)
observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(10,), dtype=np.float32)
action_space = spaces.Discrete(9)

replay_buffer = ReplayBuffer(
    buffer_size=50000,
    observation_space=observation_space,
    action_space=action_space,
    device="cpu",
    handle_timeout_termination=False,
)

# 2. Resolve episode files (KaggleHub first, local replays/ fallback)
json_files = []
if latest_episodes:
    try:
        print(f"Downloading dataset: kaggle/{latest_episodes}")
        episodes_path = kagglehub.dataset_download(f"kaggle/{latest_episodes}")
        json_files = glob.glob(f"{episodes_path}/*.json")
    except Exception as exc:
        print(f"KaggleHub download failed ({exc}); falling back to local replays/.")

if not json_files:
    json_files = sorted(glob.glob("replays/*.json"))

print(f"Found {len(json_files)} episode files. Parsing...")

action_counts = {i: 0 for i in range(9)}

# 3. Seed replay buffer from expert episodes (player-aware)
for file in json_files[:20]:
    try:
        with open(file, "r") as f:
            data = json.load(f)

        steps = data.get("steps", [])
        if len(steps) < 2:
            continue

        for i in range(len(steps) - 1):
            step_row = steps[i][0]
            next_row = steps[i + 1][0]

            current_state = step_row.get("observation", {})
            next_state = next_row.get("observation", {})
            action = step_row.get("action")
            if action is None:
                continue

            player_id = obs_to_dict(current_state).get("player", 0)
            action_idx = parse_action_from_replay(action)
            action_counts[action_idx] += 1

            reward = next_row.get("reward", 0.0)
            if reward is None:
                reward = 0.0

            done = i == len(steps) - 2

            replay_buffer.add(
                obs=parse_obs(current_state, player_id),
                next_obs=parse_obs(next_state, player_id),
                action=to_replay_action(action_idx),
                reward=np.array([reward], dtype=np.float32),
                done=np.array([done], dtype=np.float32),
                infos=[{}],
            )
    except Exception as exc:
        print(f"Error parsing {file}: {exc}")

non_pass = sum(v for k, v in action_counts.items() if k != 0)
print(f"Success! Replay Buffer seeded with {replay_buffer.pos} expert transitions.")
print("Action distribution:", {ACTION_NAMES[k]: v for k, v in action_counts.items() if v})
print(f"Non-PASS mapping rate: {non_pass / max(replay_buffer.pos, 1):.1%}")

In [ ]:
# Inspect the first transition in the replay buffer to verify parsing
if replay_buffer.pos == 0:
    raise RuntimeError("Replay buffer is empty. Check dataset download or local replays/.")

print("--- First Transition ---")
print(f"Observation: {replay_buffer.observations[0]}")
print(f"Action: {replay_buffer.actions[0]} -> {ACTION_NAMES[int(replay_buffer.actions[0])]}")
print(f"Reward: {replay_buffer.rewards[0]}")
print(f"Next Observation: {replay_buffer.next_observations[0]}")
print(f"Done: {replay_buffer.dones[0]}")

## Network Architecture & Pre-training

Unified Dueling Q-Network with a single 9-action head. Offline phase uses behavioral cloning on expert actions; online phase uses Double DQN fine-tuning with the same weights.

In [ ]:
class DuelingDQN(nn.Module):
    def __init__(self, state_dim, action_dim=9):
        super().__init__()
        self.shared_net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        features = self.shared_net(x)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return value + advantages - advantages.mean(dim=-1, keepdim=True)

print("Network architecture defined successfully.")

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

state_dim = 10
action_dim = 9
batch_size = 128

policy_net = DuelingDQN(state_dim, action_dim).to(device)
target_net = DuelingDQN(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
bc_epochs = 500

print(f"Starting offline behavioral cloning for {bc_epochs} iterations...")
losses = []

policy_net.train()
for epoch in range(bc_epochs):
    batch = replay_buffer.sample(min(batch_size, replay_buffer.pos))
    states = batch.observations.to(device)
    actions = batch.actions.to(device).long().squeeze(-1)

    logits = policy_net(states)
    loss = F.cross_entropy(logits, actions)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"BC Iteration {epoch + 1}/{bc_epochs} - Avg Loss (last 100): {np.mean(losses[-100:]):.4f}")

target_net.load_state_dict(policy_net.state_dict())
print("Offline behavioral cloning completed!")

In [ ]:
torch.save(policy_net.state_dict(), "model.pth")
print("Model weights saved to model.pth")

## Online Fine-Tuning

Fine-tune the BC-initialized Dueling DQN against the live Gymnasium wrapper using the same Discrete(9) action space.

In [ ]:
class KaggricultureOnlineEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = make("kaggriculture", debug=False)
        self.trainer = self.env.train([None, "random"])
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(10,), dtype=np.float32)
        self.action_space = spaces.Discrete(9)

    def reset(self, seed=None, options=None):
        obs = self.trainer.reset()
        obs_dict = obs_to_dict(obs)
        return parse_obs(obs_dict, obs_dict.get("player", 0)), {}

    def step(self, action):
        action_dict = format_action_for_env(action)
        obs, reward, done, info = self.trainer.step(action_dict)
        if done is None:
            done = True
        if reward is None:
            reward = 0.0
        obs_dict = obs_to_dict(obs)
        return parse_obs(obs_dict, obs_dict.get("player", 0)), reward, done, False, {}

online_env = KaggricultureOnlineEnv()
print("Online Environment initialized successfully.")

In [ ]:
fine_tune_episodes = 5
epsilon = 0.1
gamma = 0.99

print(f"Starting online fine-tuning for {fine_tune_episodes} episodes...")

policy_net.train()
for ep in range(fine_tune_episodes):
    obs, _ = online_env.reset()
    done = False
    total_reward = 0.0
    step_count = 0

    while not done:
        if random.random() < epsilon:
            action = online_env.action_space.sample()
        else:
            state_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                q_vals = policy_net(state_tensor)
            action = int(q_vals.argmax(dim=1).item())

        next_obs, reward, done, _, _ = online_env.step(action)
        total_reward += reward

        replay_buffer.add(
            obs=obs,
            next_obs=next_obs,
            action=to_replay_action(action),
            reward=np.array([reward], dtype=np.float32),
            done=np.array([done], dtype=np.float32),
            infos=[{}],
        )
        obs = next_obs
        step_count += 1

        if replay_buffer.pos > batch_size:
            batch = replay_buffer.sample(batch_size)
            b_states = batch.observations.to(device)
            b_actions = batch.actions.to(device).long().squeeze(-1)
            b_rewards = batch.rewards.to(device).squeeze(-1)
            b_next_states = batch.next_observations.to(device)
            b_dones = batch.dones.to(device).squeeze(-1)

            current_q = policy_net(b_states).gather(1, b_actions.unsqueeze(1)).squeeze(-1)
            with torch.no_grad():
                next_q_policy = policy_net(b_next_states)
                next_q_target = target_net(b_next_states)
                best_next_actions = next_q_policy.argmax(dim=1, keepdim=True)
                next_q_val = next_q_target.gather(1, best_next_actions).squeeze(-1)
                target_q = b_rewards + gamma * (1 - b_dones) * next_q_val

            loss = F.smooth_l1_loss(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    if (ep + 1) % 2 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    buffer_size = replay_buffer.pos if not replay_buffer.full else replay_buffer.buffer_size
    print(
        f"Episode {ep + 1}/{fine_tune_episodes} | Steps: {step_count} | "
        f"Total Reward: {total_reward:.2f} | Buffer Size: {buffer_size}"
    )

print("Online fine-tuning complete!")

In [ ]:
torch.save(policy_net.state_dict(), "model.pth")
print("Fine-tuned model weights saved to model.pth")

## Generate agent.py for Kaggle Submission

Standalone submission agent using the unified DuelingDQN architecture and embedded weights.

In [ ]:
with open("model.pth", "rb") as f:
    encoded_model = base64.b64encode(f.read()).decode("utf-8")

agent_code = f"""import base64
import numpy as np
import torch
import torch.nn as nn

class DuelingDQN(nn.Module):
    def __init__(self, state_dim, action_dim=9):
        super().__init__()
        self.shared_net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        features = self.shared_net(x)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return value + advantages - advantages.mean(dim=-1, keepdim=True)

model_b64 = "{encoded_model}"
device = torch.device("cpu")
model = DuelingDQN(state_dim=10, action_dim=9).to(device)

with open("temp_model.pth", "wb") as f:
    f.write(base64.b64decode(model_b64))
model.load_state_dict(torch.load("temp_model.pth", map_location=device))
model.eval()

def obs_to_dict(obs):
    if obs is None:
        return {{}}
    if isinstance(obs, dict):
        return obs
    try:
        return dict(obs)
    except (TypeError, ValueError):
        pass
    out = {{}}
    for key in ("player", "day", "hour", "step", "farms", "market", "private", "town"):
        if hasattr(obs, key):
            out[key] = getattr(obs, key)
    return out

def parse_obs(obs, player_id=0):
    obs = obs_to_dict(obs)
    farms = obs.get("farms", [])
    if not farms or len(farms) <= player_id:
        return np.zeros(10, dtype=np.float32)

    farm = farms[player_id]
    fx, fy = farm.get("farmer", [0, 0])
    money = farm.get("money", 0)

    opp_id = 1 - player_id
    opp_money = farms[opp_id].get("money", 0) if len(farms) > opp_id else 0

    market = obs.get("market", {{}}) or {{}}
    prices = market.get("prices", {{}}) or {{}}
    melon_price = prices.get("MELON", 0)

    market_inv = market.get("inventory", {{}}) or {{}}
    melon_market_inv = market_inv.get("MELON", 0)

    private = obs.get("private", {{}}) or {{}}
    shed = private.get("shed", {{}}) or {{}}
    melon_inv = shed.get("MELON", 0)

    seeds = private.get("seeds", {{}}) or {{}}
    melon_seeds = seeds.get("MELON", 0)

    day = obs.get("day", 0)
    hour = obs.get("hour", 0)

    return np.array([
        float(fx),
        float(fy),
        float(money) / 3000.0,
        float(opp_money) / 3000.0,
        float(melon_price) / 100.0,
        float(melon_market_inv) / 100.0,
        float(melon_inv) / 10.0,
        float(melon_seeds) / 10.0,
        float(day) / 30.0,
        float(hour) / 24.0,
    ], dtype=np.float32)

def agent(obs):
    player = obs_to_dict(obs).get("player", 0)
    state = parse_obs(obs, player)
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        q_vals = model(state_tensor)
    action_idx = int(q_vals.argmax(dim=1).item())

    actions = [
        {{"farmer": ["PASS"], "hands": [], "market": []}},
        {{"farmer": ["NORTH"], "hands": [], "market": []}},
        {{"farmer": ["SOUTH"], "hands": [], "market": []}},
        {{"farmer": ["EAST"], "hands": [], "market": []}},
        {{"farmer": ["WEST"], "hands": [], "market": []}},
        {{"farmer": ["WATER"], "hands": [], "market": []}},
        {{"farmer": ["HARVEST"], "hands": [], "market": []}},
        {{"farmer": ["PLANT", "MELON"], "hands": [], "market": [["BUY_SEED", "MELON", 1]]}},
        {{"farmer": ["PASS"], "hands": [], "market": [["SELL", "MELON", 1]]}},
    ]
    return actions[action_idx]
"""

with open("agent.py", "w") as f:
    f.write(agent_code)
print("Standalone agent.py successfully generated!")

## Verification

Smoke test: run the generated agent against the built-in `random` baseline.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("submission_agent", "agent.py")
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)

test_env = make("kaggriculture", debug=False)
rewards = test_env.run([submission_agent.agent, "random"])
print("Smoke test rewards (agent, random):", rewards)
print("Agent completed a full match against random baseline.")